# 实验三 · QQ 邮箱分步练习

实验报告在同目录。这份练习本按小格学习实验三的前两个阶段。Mac 或 Windows 的第一次启动见 [练习本使用说明](PRACTICE.md)。

每次运行一个代码格。本次已讲到第 10 步，换电脑后先在新内核按顺序完成前置格，再从第 11 步（一）接着学。

## 第 1 步｜启动 Playwright

在 Jupyter 当前的 Python 3 内核启动 Playwright。后面的代码用变量 p 控制浏览器。

In [ ]:
from playwright.async_api import async_playwright
p = await async_playwright().start()
print("Playwright 已启动")

## 第 2 步｜打开 Chromium

Mac 原生运行时会打开独立的 Chromium 窗口。课程 Docker 环境则会把浏览器显示在 noVNC。

In [ ]:
import sys
from pathlib import Path

visual_dir = Path("../L07_L08_visual").resolve()
if (visual_dir / "course_visual.py").is_file():
    sys.path.insert(0, str(visual_dir))
    import course_visual

    browser = await course_visual.launch(p, headless=False)
else:
    browser = await p.chromium.launch(headless=False)

print("Chromium 已打开")

## 第 3 步｜创建浏览会话和标签页

context 管理浏览会话，page 指向单独的标签页。about:blank 表示空白页。

In [ ]:
context = await browser.new_context()
page = await context.new_page()
print(page.url)

## 第 4 步｜打开 QQ 邮箱网页

goto() 打开指定网址。domcontentloaded 表示页面的基本结构已加载。

In [ ]:
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print(await page.title())

## 第 5 步｜扫码进入收件箱

在实际浏览器窗口用手机 QQ 扫码并确认。程序等待中英文收件箱入口可见，再点击它。

In [ ]:
import re

inbox = page.locator(".sidebar-menu-text").filter(
    has_text=re.compile(r"^\s*(收件箱|Inbox)\s*$", re.IGNORECASE)
).first

print("请在 QQ 邮箱窗口用手机扫码，最多等 10 分钟。", flush=True)
await inbox.wait_for(state="visible", timeout=600_000)
await inbox.click()
print("已点击收件箱")

## 第 6 步｜统计当前加载的邮件行

先等第一行可见，然后用 count() 读取当前加载的行数。

In [ ]:
rows = page.locator(".mail-list-page-item[data-mailid]")
await rows.first.wait_for(state="visible", timeout=15_000)
print("当前页面加载的邮件行数", await rows.count())

## 第 7 步（一）｜找到一封普通邮件

has_not 会筛掉含广告聚合标记的邮件行。first 取过滤后的第一条。

In [ ]:
ordinary_rows = rows.filter(has_not=page.locator(".gg-unread-count"))
row = ordinary_rows.first
await row.wait_for(state="visible", timeout=15_000)
print("找到第一封普通邮件")

## 第 7 步（二）｜读取邮件 ID 和发件人

get_attribute() 读属性，inner_text() 读文字。两个字段都从同一行读取。

In [ ]:
mail_id = await row.get_attribute("data-mailid")
sender = (await row.locator(".cmp-account-nick").first.inner_text()).strip()
print("邮件 ID", bool(mail_id), "发件人", bool(sender))

## 第 7 步（三）｜读取主题和时间

主题和时间的定位器都以 row 为范围。输出只显示字段是否有内容。

In [ ]:
subject = (await row.locator(".mail-subject").first.inner_text()).strip()
mail_time = (await row.locator(".mail-time").first.inner_text()).strip()
print("主题", bool(subject), "时间", bool(mail_time))

## 第 8 步（一）｜悬停显示邮箱卡片

把鼠标移到时间上，等旧卡片隐藏。再悬停发件人姓名的外层区域。邮箱卡片在整页中定位。

In [ ]:
async def open_sender_card(mail_row):
    card = page.locator(".cmp-email-address:visible").first
    await mail_row.locator(".mail-time").first.hover()
    await card.wait_for(state="hidden", timeout=15_000)

    trigger = mail_row.locator(".cmp-account-wrap").first
    await trigger.scroll_into_view_if_needed()
    await page.locator(".ui-float-scroll-body-scrolling").first.wait_for(state="hidden")
    await trigger.hover()
    await card.wait_for(state="visible", timeout=15_000)
    return card

email_card = await open_sender_card(row)
print("发件人邮箱卡片已显示")

## 第 8 步（二）｜从卡片读邮箱地址

逐个查显示文字、title 和 href。找到地址后把它存到 email，输出只显示是否读到。

In [ ]:
import re

async def read_sender_email(card):
    candidates = [
        await card.inner_text(),
        await card.get_attribute("title"),
        await card.get_attribute("href"),
    ]
    for text in candidates:
        match = re.search(
            r"[A-Za-z0-9.!#$%&'*+/=?^_\x60{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+",
            text or "",
        )
        if match:
            return match.group()
    raise ValueError("邮箱卡片已出现，但没有读取到有效地址。")

email = await read_sender_email(email_card)
print("已读到邮箱地址", bool(email))

## 第 9 步｜把一封邮件整理成字典

字典以字段名为键。内部 ID 可以用于定位，基础表另外保存四个字段。

In [ ]:
record = {
    "邮件ID": mail_id,
    "发件人": sender,
    "邮箱": email,
    "主题": subject,
    "时间": mail_time,
}
print("已整理字段", list(record))


## 第 10 步｜把读一封邮件写成函数

第一个函数复用取文字操作，第二个函数接收邮件行，返回一封邮件的字典。

In [ ]:
async def read_row_text(mail_row, selector):
    return (await mail_row.locator(selector).first.inner_text()).strip()

async def read_practice_row(mail_row):
    return {
        "邮件ID": await mail_row.get_attribute("data-mailid"),
        "发件人": await read_row_text(mail_row, ".cmp-account-nick"),
        "邮箱": await read_sender_email(await open_sender_card(mail_row)),
        "主题": await read_row_text(mail_row, ".mail-subject"),
        "时间": await read_row_text(mail_row, ".mail-time"),
    }

## 第 11 步（一）｜按 ID 找回邮件并设置滚动位置

本次接续点。根据 data-mailid 重新找行。set_scroll_top() 会给浏览器两次绘制机会来更新列表。

In [ ]:
import asyncio
import json

ROW_SELECTOR = ".mail-list-page-item[data-mailid]"
scroller = page.locator(".mail-list-page .ui-float-scroll-body").first

def get_mail_row(current_id):
    value = json.dumps(current_id, ensure_ascii=False)
    return page.locator(f"{ROW_SELECTOR}[data-mailid={value}]").first

async def set_scroll_top(top):
    await scroller.evaluate("(box, value) => {box.scrollTop = value;}", top)
    await page.evaluate("() => new Promise(r => requestAnimationFrame(() => requestAnimationFrame(r)))")

## 第 11 步（二）｜取当前屏幕的邮件 ID

浏览器比较邮件行和滚动区域的位置，只返回与可见区域相交的行的 ID。

In [ ]:
async def visible_mail_ids():
    return await scroller.evaluate("""(box, selector) => {
      const bounds = box.getBoundingClientRect();
      const low = Math.max(bounds.top, 0);
      const high = Math.min(bounds.bottom, innerHeight);
      return [...box.querySelectorAll(selector)].filter(row => {
        const r = row.getBoundingClientRect();
        return r.height > 0 && r.bottom > low + 1 && r.top < high - 1;
      }).map(row => row.getAttribute("data-mailid")).filter(Boolean);
    }""", ROW_SELECTOR)

async def scroll_metrics():
    return await scroller.evaluate("""box => ({
      top: box.scrollTop, height: box.clientHeight,
      end: box.scrollHeight - box.clientHeight
    })""")

## 第 11 步（三）｜移到下一屏

滚动约一屏并留少量重叠。到末尾返回 False。移动后等 ID 更新再继续。

In [ ]:
async def next_mail_screen(metrics, old_ids):
    top = metrics["top"]
    target = min(top + max(1, metrics["height"] - 8), metrics["end"])
    if target <= top + 1:
        return False

    await set_scroll_top(target)
    for attempt in range(35):
        new_ids = await visible_mail_ids()
        if new_ids and new_ids != old_ids:
            return True
        await asyncio.sleep(0.1)
    raise RuntimeError("邮件列表滚动后尚未更新，请核对滚动容器。")

## 第 11 步（四）｜采集当前屏幕的普通邮件

循环一轮处理一个 ID。已有 ID 和广告行跳过。append() 把一封读好的邮件放入列表。

In [ ]:
async def collect_screen(ids, top, records, positions, seen):
    for current_id in ids:
        if current_id in seen:
            continue
        current_row = get_mail_row(current_id)
        if await current_row.locator(".gg-unread-count").count():
            continue
        records.append(await read_practice_row(current_row))
        positions[current_id] = top
        seen.add(current_id)
        if len(records) == 10:
            break

## 第 11 步（五）｜从顶部累计采集 10 封

从顶部开始，当前一屏不足 10 封时再滚动。到了底部就显示实际采集数。

In [ ]:
basic_records, practice_positions, seen_ids = [], {}, set()
await set_scroll_top(0)

while len(basic_records) < 10:
    metrics = await scroll_metrics()
    ids = await visible_mail_ids()
    await collect_screen(ids, metrics["top"], basic_records, practice_positions, seen_ids)
    if len(basic_records) == 10 or not await next_mail_screen(metrics, ids):
        break

print("已采集普通邮件", len(basic_records), "封，目标 10 封")

## 第 12 步｜用 pandas 保存 CSV

按四列整理记录。index=False 让 CSV 只保留指定字段。原始练习文件写入本机 private_data/practice/。

In [ ]:
from pathlib import Path
import pandas as pd

BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
table = pd.DataFrame(basic_records, columns=BASIC_COLUMNS)
folder = Path("private_data") / "practice"
folder.mkdir(parents=True, exist_ok=True)
csv_path = folder / "mail_basic_practice.csv"
table.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("已写入练习 CSV", len(table), "行")

## 第 13 步｜回读并确认行列

read_csv() 把文件读回来。shape 给出行数和列数，输出还会显示四个列名的顺序是否正确。

In [ ]:
csv_data = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
row_count, column_count = csv_data.shape
print("CSV 回读", row_count, "行 ×", column_count, "列")
print("四列名称和顺序正确", list(csv_data.columns) == BASIC_COLUMNS)
assert row_count == len(basic_records)
assert list(csv_data.columns) == BASIC_COLUMNS

## 第 14 步｜回网页核对第 1、5、10 封

Python 的行号从 0 开始。按保存时记录的滚动位置和邮件 ID 找回同一封，再与 CSV 四项逐个比较。

In [ ]:
sample_indices = [i for i in (0, 4, 9) if i < len(csv_data)]
for index in sample_indices:
    current_id = basic_records[index]["邮件ID"]
    await set_scroll_top(practice_positions[current_id])
    fresh = await read_practice_row(get_mail_row(current_id))
    checks = [fresh[key] == csv_data.iloc[index][key] for key in BASIC_COLUMNS]
    print(f"第 {index + 1} 封", *checks)

## 结束练习

完成后关闭 Chromium 和 Playwright。下一次练习从第 1 格开始。

In [ ]:
await browser.close()
await p.stop()